# ComCam Tracker Error Analysis and Visualization

This notebook provides a step-by-step guide to working with ComCam data. It covers the entire process, from downloading the data to performing error analysis and visualization. Specifically, it will:

- Download Star Tracker data from Rubin TV, selecting specific observation dates.
- Transfer the data to USDF for further processing.
- Read and extract information from the downloaded files.
- Analyze and visualize azimuth errors to assess the accuracy of Star Tracker observations.


## Downloading ComCam Data  

To analyze the stars tracked, the first step is to access Rubin TV for download the using the following link:  
[Rubin TV - Summit USDF](https://usdf-rsp.slac.stanford.edu/rubintv/summit-usdf)  

Once on the Rubin TV page, you can access data from **ComCam** by clicking on the corresponding icon.  

### Selecting a Specific Date  

To search for data from a specific observation day, click on the **Historical** botton.  

You can use the calendar to select the year, month, and day of the data you want to analyze. Suppose we want to download data from **November 09, 2024**. Click on the calendar to select this date.  

Once the date is selected, the available data for that day will be displayed.  

### Downloading the Data  

To download the data, click on **Download Metadata**, and a `.json` file containing the information will be downloaded. In this case, the file will be named: comcam_2024-11-09.json

## Transferring Data to USDF  

Now that the data is stored on your local computer, you need to copy it to USDF to be able to use it.  

### Uploading Data to USDF  

To do this, you can create a folder named **ComCam** inside your `notebooks/` directory and upload the data there. However, you can also place the data in any other directory of your choice.  

### Copying Data to Scratch Directory  

For better accessibility and performance, it is recommended to store the data in the **scratch directory**. You can copy the uploaded files to:  `/scratch/users/<your_username>/ComCam`

Before doing so, make sure that the `ComCam` folder exists inside your user directory. If it does not exist, create it first using:  

```bash
mkdir -p /scratch/users/<your_username>/ComCam
```

Now, you can copy the files from notebooks/ComCam/ to the scratch directory by running the following command in the terminal:

```bash
cp /home/<your_username>/notebooks/ComCam/* /scratch/users/<your_username>/ComCam/
```

This will move all the files you uploaded into the starTracker folder in the scratch directory, making them ready for analysis.

We are now ready to start the analysis

In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

## Reading and Processing Star Tracker Data

In [ ]:
# Modify this path to your own directory where the data is stored  
# Example: DATA_DIR = "/scratch/users/<your_username>/ComCam"
DATA_DIR = "/scratch/users/t/toribio/ComCam"

def read_rubin_tv_json(day_obs, camera):
    """
    Reads a RubinTV JSON file for the given observation day and camera.

    Parameters:
    ----------
    day_obs : int
        Observation day in the format YYYYMMDD.
    camera : str
        Camera type: 'ComCam'.

    Returns:
    -------
    pd.DataFrame
        Transposed DataFrame containing the JSON data.
    """
    # Convert day_obs to datetime and then to astropy Time
    time_obj = Time(datetime.strptime(str(day_obs), "%Y%m%d"))
    date_str = time_obj.strftime("%Y-%m-%d")

    # Define filename based on camera type
    filename_map = {
        "ComCam": f"{DATA_DIR}/comcam_{date_str}.json",
        "Wide": f"{DATA_DIR}/startracker_wide_{date_str}.json",
        "Narrow": f"{DATA_DIR}/startracker_narrow_{date_str}.json",
        "AuxTel": f"{DATA_DIR}/auxtel_{date_str}.json",
        "Fast": f"{DATA_DIR}/startracker_fast_{date_str}.json",
    }

    filename = filename_map.get(camera)

    if not filename:
        raise ValueError(f"Invalid camera type: {camera}")

    try:
        df = pd.read_json(filename).transpose()
        print(f"Loaded file: {filename}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found - {filename}")
        return None

## Visualise the data

> **Important Note**  
> This notebook cannot be executed for dates earlier than **October 23, 2024**, as the required metadata is not available for those dates.  
> 
> Additionally, analysis involving **azimuth (Az)** and **altitude (Alt)** cannot be performed because the corresponding data is not included in the metadata and LSSTCam is already installed.  
> 
> Please keep this in mind when selecting observation dates or attempting to analyze Alt/Az information.


In [ ]:
# List of observation dates in YYYYMMDD format
# Modify this list with your own data
dates = [
    20241130, 20241109, 20241102, 20241101,
    20241031, 20241030, 20241029, 20241028, 20241027, 20241026, 20241025, 20241024
]

# Select the type of camera (in this case ComCam)
camera = "ComCam" 

# Variables to store results
x_axis = []
delta_azs = []
delta_els = []
delta_ras = []
delta_decs = []
counter = 1

for day_obs in dates:

    # Read data from RubinTV JSON files
    df = read_rubin_tv_json(day_obs, camera)
    if df is None:
        continue  # Skip if the file does not exist

    # Remove rows with NaN values
    df = df.dropna(subset=['delta Ra (arcsec)', 'delta Dec (arcsec)'] )
    
    for seq_num, row in df.iterrows():
        if camera == "ComCam":
            ra_col = "delta Ra (arcsec)"
            dec_col = "delta Dec (arcsec)"
        else:
            continue 
            
        processed = False
        
        if ra_col in df.columns:
           delta_ra = row[ra_col]
           delta_ra = float(delta_ra)
           delta_ras.append(abs(delta_ra))
           processed = True
          
        if dec_col in df.columns:
           delta_dec = row[dec_col]
           delta_dec = float(delta_dec)
           delta_decs.append(abs(delta_dec))
           processed = True

        x_axis.append(counter)

    
    counter += 1


## RA Error Trend Plot

In [ ]:
# Extend dates list to include the last observation date
plot_dates = dates

# Define x-axis ticks
x_ticks = np.arange(1, len(plot_dates) + 1, 1)

# Create the plot for StarTracker Narrow Azimuth Error Trend
plt.figure(figsize=(8, 6))
plt.title("ComCam RA Error Trend")

# Scatter plot of ra errors
plt.scatter(x_axis, delta_ras, color="blue", alpha=0.7, label="Azimuth Error")

# Use logarithmic scale for the y-axis
plt.yscale("log")
plt.ylim(1.0, 1.0e5)

# Set x-axis labels with proper rotation
plt.xticks(x_ticks, plot_dates, rotation=-45)

# Label axes
plt.xlabel("Date")
plt.ylabel("RA Error (arcseconds)")

# Show grid and legend for better readability
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.tight_layout()

# Display the plot
plt.show()


## Dec Error Trend Plot

In [ ]:
# Extend dates list to include the last observation date
plot_dates = dates

# Define x-axis ticks
x_ticks = np.arange(1, len(plot_dates) + 1, 1)

# Create the plot for StarTracker Narrow Azimuth Error Trend
plt.figure(figsize=(8, 6))
plt.title("ComCam Dec Error Trend")

# Scatter plot of dec errors
plt.scatter(x_axis, delta_decs, color="blue", alpha=0.7, label="Dec Error")

# Use logarithmic scale for the y-axis
plt.yscale("log")
plt.ylim(1.0, 1.0e5)

# Set x-axis labels with proper rotation
plt.xticks(x_ticks, plot_dates, rotation=-45)

# Label axes
plt.xlabel("Date")
plt.ylabel("Dec Error (arcsec)")

# Show grid and legend for better readability
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.tight_layout()

# Display the plot
plt.show()